# 05 — Numerical validation and visualization

Treat the PINN as a numerical approximation and measure it with independent field, residual, initial-condition, boundary-condition, and norm-based diagnostics.

In [ ]:
import torch
import matplotlib.pyplot as plt
from pinn import MLP,PINNConfig,PINNTrainer,sample_heat_equation,heat_exact_solution,heat_residual
torch.set_default_dtype(torch.float32)
alpha=0.1
points=sample_heat_equation(3000,600,600,seed=21)
trainer=PINNTrainer(MLP(hidden_dim=48,hidden_layers=3),PINNConfig(alpha=alpha,initial_weight=20,boundary_weight=20,epochs=1200,log_every=400,seed=21))
history=trainer.train(points)


## 1. Structured evaluation grid

In [ ]:
nx,nt=201,121
x=torch.linspace(-1,1,nx); t=torch.linspace(0,1,nt)
xx,tt=torch.meshgrid(x,t,indexing='ij')
grid=torch.stack([xx.reshape(-1),tt.reshape(-1)],1)
with torch.no_grad(): pred=trainer.predict(grid).reshape(nx,nt)
exact=heat_exact_solution(grid[:,0:1],grid[:,1:2],alpha).reshape(nx,nt)
error=pred-exact
print('RMSE',float(torch.sqrt(torch.mean(error.square()))))
print('MAE',float(error.abs().mean()))
print('Linf',float(error.abs().max()))


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,field,title in zip(axes,[pred,exact,error.abs()],['PINN solution','Analytical solution','Absolute error']):
    im=ax.imshow(field.detach().numpy(),origin='lower',aspect='auto',extent=[0,1,-1,1])
    ax.set_xlabel('t'); ax.set_ylabel('x'); ax.set_title(title); fig.colorbar(im,ax=ax)
plt.tight_layout(); plt.show()


## 2. Error slices

Global error norms can hide localized failures. Inspect fixed-time and fixed-space slices.

In [ ]:
plt.figure(figsize=(8,4))
for tau in [0,.25,.5,.75,1]:
    j=round(tau*(nt-1)); plt.plot(x.numpy(),error[:,j].detach().numpy(),label=f't={tau:.2f}')
plt.xlabel('x'); plt.ylabel('prediction - exact'); plt.legend(); plt.show()


In [ ]:
plt.figure(figsize=(8,4))
for xi in [-1,-.5,0,.5,1]:
    i=round((xi+1)*(nx-1)/2); plt.plot(t.numpy(),error[i,:].detach().numpy(),label=f'x={xi:.2f}')
plt.xlabel('t'); plt.ylabel('prediction - exact'); plt.legend(); plt.show()


## 3. Independent PDE-residual test set

In [ ]:
test=sample_heat_equation(5000,200,200,seed=999)
r=heat_residual(trainer.model,test.interior.clone().requires_grad_(True),alpha)
print('test PDE RMSE',float(torch.sqrt(torch.mean(r.square()))))
print('test PDE Linf',float(r.abs().max()))


## 4. Initial and boundary conditions

In [ ]:
with torch.no_grad():
    u0=trainer.predict(test.initial)
    left=trainer.predict(test.left_boundary)
    right=trainer.predict(test.right_boundary)
target0=torch.sin(torch.pi*test.initial[:,0:1])
print('initial RMSE',float(torch.sqrt(torch.mean((u0-target0).square()))))
print('left BC RMSE',float(torch.sqrt(torch.mean(left.square()))))
print('right BC RMSE',float(torch.sqrt(torch.mean(right.square()))))


## 5. Relative norms

In [ ]:
print('relative L2',float(torch.linalg.vector_norm(error.reshape(-1))/torch.linalg.vector_norm(exact.reshape(-1))))
print('relative Linf',float(error.abs().max()/exact.abs().max()))
